# Code Critic — Architecture Description
**Author**: Jayaprakash Sivanandam &nbsp;|&nbsp; **Date**: 2026-04-29

This notebook describes the implemented architecture for the Code Critic agentic AI application.
It supersedes the earlier LangGraph-based design documented in `PROJECT_DESCRIPTION.ipynb`.

## Overview

Code Critic is an agentic SQL code reviewer for Snowflake. It accepts a stored procedure name or a raw SQL query, runs it through a multi-agent analysis pipeline, and returns a structured `FindingsReport` covering performance, security, and style.

The system is built on the **A2A (Agent-to-Agent) SDK**. Each agent is an independent HTTP service that communicates via the A2A protocol. Agents do not share state — they receive a task, call an LLM, and return a typed JSON response.

## Agent Pipeline

```
User Input
    │
    ▼
┌─────────────────────────────────────────────────────────┐
│  Orchestrator  (Claude Opus · port 8016)                │
│                                                         │
│  1. call_router()          →  Router (8011)             │
│  2. call_schema_fetcher()  →  Schema-Fetcher (8012)     │
│     (only if stored_procedure)                          │
│                                                         │
│  3. asyncio.gather():                                   │
│       call_perf_analyzer()    → Perf Analyzer (8013)    │
│       call_security_auditor() → Security Auditor (8014) │
│       call_style_reviewer()   → Style Reviewer (8015)   │
│                                                         │
│  4. Opus synthesizes → FindingsReport                   │
└─────────────────────────────────────────────────────────┘
    │
    ▼
FindingsReport (JSON)
```

## Agent Registry

| Port | Agent | Model | Role |
|------|-------|-------|------|
| 8011 | `router` | Gemma | Classify input as `sql_query`, `stored_procedure`, or `unknown` |
| 8012 | `schema-fetcher` | Gemma | Fetch Snowflake DDL via Snowpark; extract executable ETL body |
| 8013 | `perf-analyzer` | Gemma | Detect performance inefficiencies; runs in parallel |
| 8014 | `security-auditor` | Gemma | Flag security vulnerabilities; runs in parallel |
| 8015 | `style-reviewer` | Gemma | Catch style and readability violations; runs in parallel |
| 8016 | `orchestrator` | Claude Opus | Drive full pipeline; synthesize `FindingsReport` |

Gemma (`gemma-4-26b-a4b-it`) is used for the specialist agents because it is fast and cost-effective for focused, single-dimension analysis. Claude Opus (`claude-opus-4-6`) is used for the orchestrator because it handles multi-step reasoning, agent coordination, and final synthesis in one step.

## Data Models

All inter-agent messages are typed Pydantic models serialized as JSON.

### Router output
```python
class RouterDecision(BaseModel):
    input_type:  InputType        # sql_query | stored_procedure | unknown
    confidence:  float
    reasoning:   str
    object_name: Optional[str]    # fully-qualified procedure name, if applicable
```

### Schema-fetcher output
```python
class SchemaFetchResult(BaseModel):
    object_name: str
    object_type: str              # always PROCEDURE in current scope
    sql_text:    str              # executable ETL body extracted from DDL
    source:      str              # snowflake_ddl_code
```

### Analyzer output (shared by all three analyzers)
```python
class Finding(BaseModel):
    severity:    Severity         # high | medium | low | info
    description: str
    snippet:     Optional[str]    # offending SQL fragment
    suggestion:  str

class AnalysisResult(BaseModel):
    findings: list[Finding]
```

### Final orchestrator output
```python
class FindingsReport(BaseModel):
    input_type:           str
    object_name:          Optional[str]
    performance_findings: list[Finding]
    security_findings:    list[Finding]
    style_findings:       list[Finding]
    rewritten_sql:        Optional[str]   # Opus-generated improved version
    summary:              str             # Opus-generated plain-language summary
```

## Orchestrator as Synthesizer

The orchestrator handles both coordination and synthesis — there is no separate synthesizer agent. After collecting the three `AnalysisResult` payloads, it calls Opus once with `SYNTHESIZER_SKILL` as the system prompt and a combined JSON payload as the human message. Opus returns a `SynthesisOutput` containing the `summary` and `rewritten_sql`. The orchestrator then assembles the final `FindingsReport` by passing the analyzer findings through directly.

This keeps the design at six agents instead of seven and avoids an extra network hop for data that is already structured.

## Key Design Patterns

### A2A agent pattern
Every agent follows the same two-part structure:
1. **Executor class** — inherits `AgentExecutor`, implements `execute()` with an LLM chain using `with_structured_output(PydanticModel)`
2. **Server bootstrap** — `AgentCard` + `DefaultRequestHandler` + `FastAPI` app running in a background `threading.Thread` via `uvicorn`

### Structured output
All LLM calls use LangChain's `with_structured_output()` to bind a Pydantic model to the LLM response. This eliminates raw JSON parsing and gives type-safe results at each step.

### Parallel analysis
The three analyzer agents are called concurrently inside the orchestrator's `execute()` coroutine using `asyncio.gather()`. Because the orchestrator runs inside uvicorn's asyncio event loop, all three HTTP calls are dispatched simultaneously, cutting wall-clock time roughly to the latency of the slowest analyzer.

### Prompt management
All system prompts live in `skills/*/SKILL.md` files and are loaded at import time by `prompts.py`. No prompt strings appear inline in the notebook — every agent references a named constant (`ROUTER_SKILL`, `PERFORMANCE_ANALYZER_SKILL`, etc.).

## Tech Stack

| Layer | Technology |
|-------|------------|
| Agent framework | A2A SDK |
| LLM integration | LangChain (`langchain-anthropic`, `langchain-google-genai`) |
| LLMs | Gemma `gemma-4-26b-a4b-it` (Google AI Studio), Claude Opus `claude-opus-4-6` (Anthropic) |
| HTTP server | FastAPI + uvicorn |
| Data validation | Pydantic v2 |
| Snowflake access | Snowflake Snowpark Python (read-only) |
| Development format | Jupyter notebooks |
| MCP server | FastMCP (exposes Snowflake tools to Claude Code) |

## Notebook Structure

All implementation lives in `notebooks/code_critic.ipynb`. The notebook is organized as sequential sections — each agent occupies one cell — so individual agents can be re-run or modified without restarting the kernel.

| Section | Cells | Description |
|---------|-------|-------------|
| Setup | 1–2 | Pip install, imports, model constants, LLM instances |
| Router | 3–4 | `RouterAgentExecutor` + server on port 8011 |
| Schema-fetcher | 5–6 | Snowflake tools + `SchemaFetcherAgentExecutor` + server on port 8012 |
| Analyzer models | 7 | `Severity`, `Finding`, `AnalysisResult`, `FindingsReport` |
| Perf analyzer | 8 | `PerformanceAnalyzerAgentExecutor` + server on port 8013 |
| Security auditor | 9 | `SecurityAuditorAgentExecutor` + server on port 8014 |
| Style reviewer | 10 | `StyleReviewerAgentExecutor` + server on port 8015 |
| Client helpers | 11–12 | `call_router`, `call_schema_fetcher`, `call_perf_analyzer`, etc. |
| Orchestrator | 13 | `OrchestratorAgentExecutor` + server on port 8016 |
| E2E test | 14 | `call_orchestrator()` + end-to-end test against a live procedure |
| Cleanup | 15 | Stop all six servers |